In [ ]:
import pandas as pd
import glob

# Rutas de los archivos extraídos en /data
path_households = '../data/informations_households.csv'
path_weather = '../data/weather_hourly_darksky.csv'
path_holidays = '../data/uk_bank_holidays.csv'
path_acorn = '../data/acorn_details.csv'

def check_quality(df, name):
    print(f"--- Ensayo de Material: {name} ---")
    print(f"Filas: {len(df)} | Columnas: {list(df.columns)}")
    print(f"Nulos por columna:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"Duplicados totales: {df.duplicated().sum()}")
    print("-" * 30)

# Cargamos y testeamos
df_h = pd.read_csv(path_households)
df_w = pd.read_csv(path_weather)
df_hol = pd.read_csv(path_holidays)

check_quality(df_h, "Hogares (Households)")
check_quality(df_w, "Clima (Weather)")
check_quality(df_hol, "Feriados (Holidays)")

In [ ]:
# Buscamos el primer bloque de consumo para testear
blocks = glob.glob('../data/halfhourly_dataset/halfhourly_dataset/*.csv')
print(f"Total de bloques de consumo encontrados: {len(blocks)}")

# Cargamos el primer bloque como muestra
df_sample = pd.read_csv(blocks[0])

print("--- Test de Stress: Muestra de Consumo ---")
# Punto crítico: ¿El consumo viene como número o como texto?
print(df_sample.dtypes)

# Validación de valores negativos (en energía no debería haber, salvo inyección solar)
if df_sample['energy(kWh/hh)'].apply(pd.to_numeric, errors='coerce').min() < 0:
    print("⚠️ ALERTA: Hay consumos negativos. Revisar sensores.")
else:
    print("✅ Consumos positivos: OK.")

In [ ]:
print("--- Prueba de Encastre (Integridad Referencial) ---")
ids_consumo = set(df_sample['LCLid'].unique())
ids_hogares = set(df_h['LCLid'].unique())

huerfanos = ids_consumo - ids_hogares
print(f"Medidores en la muestra: {len(ids_consumo)}")
print(f"Medidores sin datos de hogar (huérfanos): {len(huerfanos)}")

if len(huerfanos) == 0:
    print("💎 EXCELENTE: Integridad total. Todos los hogares están identificados.")